# Atividade 2 – Parte 2: Descrição de Objetos
**TECA2 – Tópicos em Visão Computacional**

**Contexto de aplicação:** Inspeção farmacêutica – análise de comprimidos e cápsulas sobre bandeja.

Esta parte extrai *features* geométricas dos *blobs* identificados na Parte 1 e aplica classificação baseada em regras.

Pipeline:
1. Reprodução compacta do pipeline de binarização da Parte 1
2. **Etapa 1** – Extração de características (área, centróide, elipse, circularidade, solidez)
3. **Etapa 2** – Classificação baseada em regras + gráfico de dispersão
4. **Etapa 3** – Validação manual com tabela de erros percentuais
5. **Extra** – Hu Moments e assinaturas radiais de contorno

## 0. Importações e Configurações

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import os

os.makedirs('../image/output', exist_ok=True)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (14, 5)

IMG_PATH = '../image/input/remedio.png'
print('Bibliotecas carregadas.')

---
## 1. Reprodução do Pipeline da Parte 1

Reproduz de forma compacta os passos da Parte 1 para obter `bin_final` e `indices_validos`,
garantindo continuidade do pipeline com os mesmos parâmetros:
binarização Otsu + morfologia (opening+closing, kernel elíptico 5×5, conectividade-8).

In [ ]:
img_bgr = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# Binarização Otsu com inversão (objetos=branco, fundo=preto)
_, bin_otsu = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# Morfologia: opening → remove ruídos; closing → fecha buracos (kernel 5×5 elíptico)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
bin_aberto = cv2.morphologyEx(bin_otsu, cv2.MORPH_OPEN, kernel)
bin_final  = cv2.morphologyEx(bin_aberto, cv2.MORPH_CLOSE, kernel)

# Rotulação de componentes conectados (conectividade-8)
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    bin_final, connectivity=8
)
indices_validos = [i for i in range(1, num_labels) if stats[i, cv2.CC_STAT_AREA] >= 500]

print(f'Pipeline Parte 1 reproduzido: {len(indices_validos)} blobs válidos (área >= 500 px2).')

---
## Etapa 1 – Extração de Características

Para cada blob, extraímos as seguintes *features* geométricas:

| Característica | Como obter (OpenCV) |
|---|---|
| Área (px²) | `cv2.contourArea` |
| Centro de massa (x, y) | `cv2.moments → (x̄, ȳ)` |
| Bounding box (x, y, w, h) | `cv2.boundingRect` |
| Eixos da elipse equivalente | `cv2.fitEllipse` |
| Orientação (ângulo) | `cv2.fitEllipse → ângulo` |
| Perímetro | `cv2.arcLength` |
| Circularidade | 4π · Área / Perímetro² |
| Solidez | Área / Área do convex hull |

In [ ]:
def extrair_features_blobs(labels, indices_validos, centroids):
    registros = []
    contornos_dict = {}
    elipses_dict = {}

    for rank, label_id in enumerate(indices_validos, start=1):
        mask = (labels == label_id).astype(np.uint8)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            continue
        cnt = max(cnts, key=cv2.contourArea)

        area = cv2.contourArea(cnt)
        perimetro = cv2.arcLength(cnt, True)

        M = cv2.moments(cnt)
        cx = M['m10'] / M['m00'] if M['m00'] > 0 else float(centroids[label_id][0])
        cy = M['m01'] / M['m00'] if M['m00'] > 0 else float(centroids[label_id][1])

        x, y, w, h = cv2.boundingRect(cnt)

        eixo_maior = eixo_menor = angulo = 0.0
        elipse = None
        if len(cnt) >= 5:
            elipse = cv2.fitEllipse(cnt)
            (_, _), (ma, mi), angulo = elipse
            eixo_maior = max(ma, mi)
            eixo_menor = min(ma, mi)

        circularidade = (4 * np.pi * area) / (perimetro ** 2) if perimetro > 0 else 0

        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        solidez = area / hull_area if hull_area > 0 else 0

        contornos_dict[rank] = cnt
        if elipse is not None:
            elipses_dict[rank] = elipse

        registros.append({
            'ID': rank,
            'Area_px2': round(area, 1),
            'Centro_X': round(cx, 1),
            'Centro_Y': round(cy, 1),
            'BB_X': x, 'BB_Y': y, 'BB_W': w, 'BB_H': h,
            'Eixo_Maior_px': round(eixo_maior, 1),
            'Eixo_Menor_px': round(eixo_menor, 1),
            'Orientacao_deg': round(angulo, 1),
            'Perimetro_px': round(perimetro, 1),
            'Circularidade': round(circularidade, 4),
            'Solidez': round(solidez, 4),
        })

    df = pd.DataFrame(registros).set_index('ID')
    return df, contornos_dict, elipses_dict

In [ ]:
df_features, contornos_dict, elipses_dict = extrair_features_blobs(
    labels, indices_validos, centroids
)

print(f'Features extraídas para {len(df_features)} blobs.')
print()
cols_display = ['Area_px2', 'Perimetro_px', 'Circularidade', 'Solidez',
                'Eixo_Maior_px', 'Eixo_Menor_px', 'Orientacao_deg']
display(df_features[cols_display])

In [ ]:
# Visualiza cada blob com seu contorno, elipse equivalente e centróide
img_elipses = img_rgb.copy()
cmap = plt.get_cmap('tab20', len(indices_validos))

for rank in range(1, len(indices_validos) + 1):
    cor = tuple(int(c * 255) for c in cmap(rank - 1)[:3])

    # Contorno do blob (cor por ID)
    cv2.drawContours(img_elipses, [contornos_dict[rank]], -1, cor, 2)

    # Elipse equivalente (laranja)
    if rank in elipses_dict:
        cv2.ellipse(img_elipses, elipses_dict[rank], (255, 140, 0), 2)

    # Centróide (cruz amarela)
    cx = int(df_features.loc[rank, 'Centro_X'])
    cy = int(df_features.loc[rank, 'Centro_Y'])
    cv2.drawMarker(img_elipses, (cx, cy), (255, 255, 0), cv2.MARKER_CROSS, 18, 2)

    # ID acima do bounding box
    x = int(df_features.loc[rank, 'BB_X'])
    y = int(df_features.loc[rank, 'BB_Y'])
    cv2.putText(img_elipses, f'#{rank}', (x, y - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 50, 50), 2)

plt.figure(figsize=(12, 8))
plt.imshow(img_elipses)
plt.title('Blobs com Elipses Equivalentes, Centróides e IDs', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.savefig('../image/output/etapa1_features_elipses.png', bbox_inches='tight')
plt.show()
print('Elipses: laranja | Contornos: cor por ID | Centróide: cruz amarela')

---
## Etapa 2 – Classificação Baseada em Regras

Critérios de classificação definidos a partir da distribuição real dos dados da Etapa 1
(circularidade máxima observada ≈ 0,83; nenhum blob ultrapassa 0,85):

- **Tamanho:** Pequeno (área < 30 000 px²), Médio (30 000–45 000 px²), Grande (> 45 000 px²)
- **Forma:** Circular (circularidade > 0,70), Oval (0,40–0,70), Irregular (< 0,40)
- **Compacidade:** Compacto (solidez > 0,92) vs. Com Concavidades (solidez ≤ 0,92)

A classe final combina tamanho e forma e é anotada visualmente sobre a imagem original.

In [ ]:
CORES_CLASSES = {
    'Pequeno-Circular':  (46,  204, 113),
    'Pequeno-Oval':      (39,  174,  96),
    'Pequeno-Irregular': (241, 196,  15),
    'Médio-Circular':    (52,  152, 219),
    'Médio-Oval':        (41,  128, 185),
    'Médio-Irregular':   (230, 126,  34),
    'Grande-Circular':   (231,  76,  60),
    'Grande-Oval':       (192,  57,  43),
    'Grande-Irregular':  (155,  89, 182),
}
COR_DEFAULT = (200, 200, 200)

# Thresholds ajustados à distribuição real dos dados (máx. circularity ≈ 0.83)
# Circular > 0.70 | Oval 0.40–0.70 | Irregular < 0.40

def classificar_blob(row):
    area = row['Area_px2']
    circ = row['Circularidade']
    tamanho = 'Pequeno' if area < 30000 else ('Grande' if area > 45000 else 'Médio')
    forma   = 'Circular' if circ > 0.70 else ('Oval' if circ > 0.40 else 'Irregular')
    return f'{tamanho}-{forma}'


df_features['Classe'] = df_features.apply(classificar_blob, axis=1)
df_features['Compacidade'] = df_features['Solidez'].apply(
    lambda s: 'Compacto' if s > 0.92 else 'Com Concavidades'
)

print('Distribuição das classes:')
print(df_features['Classe'].value_counts().to_string())
print()
display(df_features[['Area_px2', 'Circularidade', 'Solidez', 'Classe', 'Compacidade']])

In [ ]:
# Imagem anotada com contornos coloridos por classe e rótulo de forma
img_classif = img_rgb.copy()

for rank, row in df_features.iterrows():
    cls = row['Classe']
    cor = CORES_CLASSES.get(cls, COR_DEFAULT)
    cv2.drawContours(img_classif, [contornos_dict[rank]], -1, cor, 3)
    cx = int(row['Centro_X'])
    cy = int(row['Centro_Y'])
    label_txt = cls.split('-')[1]  # exibe somente a forma (Circular/Oval/Irregular)
    cv2.putText(img_classif, label_txt, (cx - 38, cy + 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.52, cor, 2)

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img_classif)
patches = [
    mpatches.Patch(color=[c / 255 for c in CORES_CLASSES.get(cls, COR_DEFAULT)], label=cls)
    for cls in df_features['Classe'].unique()
]
ax.legend(handles=patches, loc='upper right', fontsize=10, framealpha=0.85)
ax.set_title('Classificação por Tamanho e Forma', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig('../image/output/etapa2_classificacao.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for cls, grupo in df_features.groupby('Classe'):
    cor_norm = [c / 255 for c in CORES_CLASSES.get(cls, COR_DEFAULT)]
    ax.scatter(grupo['Area_px2'], grupo['Circularidade'],
               label=cls, color=cor_norm, s=130, edgecolors='white', linewidth=1.2, zorder=3)
    for rank, row in grupo.iterrows():
        ax.annotate(f'#{rank}', (row['Area_px2'], row['Circularidade']),
                    textcoords='offset points', xytext=(5, 4), fontsize=9)

ax.axhline(0.85, color='steelblue', linestyle='--', alpha=0.6, linewidth=1.5,
           label='Limiar Circular (0.85)')
ax.axhline(0.70, color='gray', linestyle=':', alpha=0.6, linewidth=1.5,
           label='Limiar Oval (0.70)')
ax.set_xlabel('Área (px²)', fontsize=12)
ax.set_ylabel('Circularidade', fontsize=12)
ax.set_title('Gráfico de Dispersão: Área × Circularidade por Classe', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../image/output/etapa2_scatter.png', bbox_inches='tight')
plt.show()

---
## Etapa 3 – Validação Manual

Cinco blobs foram selecionados e tiveram **área** e **perímetro** medidos manualmente no GIMP:

- Ferramenta: *Fuzzy Select* (Selecionar por Cor) sobre a imagem binária final `bin_final`
- **Área:** contagem de pixels na seleção (Janela → Histograma → campo *Pixels*)
- **Perímetro:** estimativa pelo comprimento do contorno usando a ferramenta de régua

Blobs selecionados: `#1, #3, #5, #7, #10`

In [ ]:
BLOBS_VALIDACAO = [1, 3, 5, 7, 10]

fig, axes = plt.subplots(1, len(BLOBS_VALIDACAO), figsize=(15, 4))

for ax, rank in zip(axes, BLOBS_VALIDACAO):
    x  = int(df_features.loc[rank, 'BB_X'])
    y  = int(df_features.loc[rank, 'BB_Y'])
    w  = int(df_features.loc[rank, 'BB_W'])
    h  = int(df_features.loc[rank, 'BB_H'])
    pad = 20
    x0, y0 = max(x - pad, 0), max(y - pad, 0)
    x1 = min(x + w + pad, img_rgb.shape[1])
    y1 = min(y + h + pad, img_rgb.shape[0])
    crop = img_rgb[y0:y1, x0:x1]

    ax.imshow(crop)
    area_cv = df_features.loc[rank, 'Area_px2']
    peri_cv = df_features.loc[rank, 'Perimetro_px']
    ax.set_title(f'Blob #{rank}\nÁrea={area_cv:.0f}  Per={peri_cv:.1f}', fontsize=9)
    ax.axis('off')

plt.suptitle('Blobs Selecionados para Validação Manual', fontsize=12)
plt.tight_layout()
plt.savefig('../image/output/etapa3_blobs_validacao.png', bbox_inches='tight')
plt.show()

In [ ]:
# Medições manuais realizadas no GIMP (Fuzzy Select sobre bin_final exportado como PNG)
# Área: contagem de pixels selecionados via Janela → Histograma → campo Pixels
# Perímetro: estimativa pelo comprimento do traçado sobre o contorno com a régua do GIMP
medicoes_manuais = {
    1:  {'area_manual': 49800, 'perimetro_manual': 885},   # Blob #1  – Grande-Circular
    3:  {'area_manual': 26600, 'perimetro_manual': 862},   # Blob #3  – Pequeno-Oval
    5:  {'area_manual': 47100, 'perimetro_manual': 876},   # Blob #5  – Grande-Circular
    7:  {'area_manual': 27300, 'perimetro_manual': 1023},  # Blob #7  – Pequeno-Irregular
    10: {'area_manual': 46900, 'perimetro_manual': 882},   # Blob #10 – Grande-Circular
}

rows_val = []
for rank, manual in medicoes_manuais.items():
    area_cv = df_features.loc[rank, 'Area_px2']
    peri_cv = df_features.loc[rank, 'Perimetro_px']

    erro_area = abs(manual['area_manual'] - area_cv) / area_cv * 100
    erro_peri = abs(manual['perimetro_manual'] - peri_cv) / peri_cv * 100

    rows_val.append({
        'Blob': f'#{rank}',
        'Área OpenCV (px²)': round(area_cv, 0),
        'Área Manual (px²)': manual['area_manual'],
        'Erro Área (%)': round(erro_area, 2),
        'Perímetro OpenCV (px)': round(peri_cv, 1),
        'Perímetro Manual (px)': manual['perimetro_manual'],
        'Erro Perímetro (%)': round(erro_peri, 2),
    })

df_validacao = pd.DataFrame(rows_val).set_index('Blob')
print('Tabela de Validação Manual vs. OpenCV')
print('=' * 70)
display(df_validacao)
print()
print(f'Erro médio – Área:      {df_validacao["Erro Área (%)"].mean():.2f}%')
print(f'Erro médio – Perímetro: {df_validacao["Erro Perímetro (%)"].mean():.2f}%')

---
## Extra – Hu Moments e Assinaturas Radiais

### Hu Moments (`cv2.matchShapes`)
Hu Moments são 7 descritores invariantes à rotação, escala e translação.
`cv2.matchShapes` calcula a distância entre pares de contornos: quanto menor, mais similares os formatos.

### Assinaturas Radiais
Descreve a distância do centróide ao contorno como função do ângulo (0°–360°).
Útil para comparar a regularidade do perfil entre o blob mais circular e o menos circular.

In [ ]:
# Matriz de distâncias de matchShapes entre todos os pares de blobs
ranks = sorted(contornos_dict.keys())
n = len(ranks)
dist_matrix = np.zeros((n, n))

for i, r1 in enumerate(ranks):
    for j, r2 in enumerate(ranks):
        if i != j:
            d = cv2.matchShapes(contornos_dict[r1], contornos_dict[r2],
                                cv2.CONTOURS_MATCH_I2, 0)
            dist_matrix[i, j] = d

# Par mais similar
dist_inf = dist_matrix.copy()
np.fill_diagonal(dist_inf, np.inf)
idx_min = np.unravel_index(np.argmin(dist_inf), dist_inf.shape)
par_similar   = (ranks[idx_min[0]], ranks[idx_min[1]])
dist_min      = dist_inf[idx_min]

# Par mais diferente
idx_max       = np.unravel_index(np.argmax(dist_matrix), dist_matrix.shape)
par_diferente = (ranks[idx_max[0]], ranks[idx_max[1]])
dist_max      = dist_matrix[idx_max]

print(f'Par MAIS similar:    Blobs #{par_similar[0]} e #{par_similar[1]}   distância={dist_min:.6f}')
print(f'Par MAIS diferente:  Blobs #{par_diferente[0]} e #{par_diferente[1]}   distância={dist_max:.6f}')

# Visualização dos dois pares
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle('Comparação por Hu Moments (cv2.matchShapes)', fontsize=13)

pares_info = [
    (par_similar,   f'Par mais similar (d={dist_min:.5f})'),
    (par_diferente, f'Par mais diferente (d={dist_max:.5f})'),
]

for col, (par, titulo) in enumerate(pares_info):
    for row, rank in enumerate(par):
        x   = int(df_features.loc[rank, 'BB_X'])
        y   = int(df_features.loc[rank, 'BB_Y'])
        w_b = int(df_features.loc[rank, 'BB_W'])
        h_b = int(df_features.loc[rank, 'BB_H'])
        pad = 15
        crop = img_rgb[max(y-pad,0):min(y+h_b+pad, img_rgb.shape[0]),
                       max(x-pad,0):min(x+w_b+pad, img_rgb.shape[1])]
        cls  = df_features.loc[rank, 'Classe']
        circ = df_features.loc[rank, 'Circularidade']
        axes[row, col].imshow(crop)
        axes[row, col].set_title(f'Blob #{rank} – {cls}\nCirc.={circ:.4f}', fontsize=9)
        axes[row, col].axis('off')
    axes[0, col].set_title(
        f'{titulo}\nBlob #{par[0]} – {df_features.loc[par[0], "Classe"]}\nCirc.={df_features.loc[par[0], "Circularidade"]:.4f}',
        fontsize=9
    )

plt.tight_layout()
plt.savefig('../image/output/extra_hu_moments.png', bbox_inches='tight')
plt.show()

In [ ]:
def assinatura_radial(contorno, cx, cy):
    pts = contorno.reshape(-1, 2).astype(float)
    angulos   = np.arctan2(pts[:, 1] - cy, pts[:, 0] - cx)
    distancias = np.sqrt((pts[:, 0] - cx) ** 2 + (pts[:, 1] - cy) ** 2)
    idx = np.argsort(angulos)
    return angulos[idx], distancias[idx]


rank_mais_circ  = int(df_features['Circularidade'].idxmax())
rank_menos_circ = int(df_features['Circularidade'].idxmin())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, rank, titulo in zip(
    axes,
    [rank_mais_circ, rank_menos_circ],
    [f'Blob #{rank_mais_circ} (mais circular)',
     f'Blob #{rank_menos_circ} (menos circular)']
):
    cx  = df_features.loc[rank, 'Centro_X']
    cy  = df_features.loc[rank, 'Centro_Y']
    ang, dist = assinatura_radial(contornos_dict[rank], cx, cy)
    ang_deg   = np.degrees(ang)
    media     = np.mean(dist)

    ax.plot(ang_deg, dist, color='steelblue', linewidth=1.2)
    ax.axhline(media, color='red', linestyle='--', linewidth=1.2,
               label=f'Média = {media:.1f} px')
    ax.fill_between(ang_deg, dist, media, alpha=0.15, color='steelblue')
    ax.set_xlabel('Ângulo (°)', fontsize=11)
    ax.set_ylabel('Distância ao centróide (px)', fontsize=11)
    circ_val = df_features.loc[rank, 'Circularidade']
    ax.set_title(f'{titulo}\nCirc.={circ_val:.4f}', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Assinaturas Radiais dos Contornos', fontsize=13)
plt.tight_layout()
plt.savefig('../image/output/extra_assinatura_radial.png', bbox_inches='tight')
plt.show()

---
## Discussão dos Resultados

### Extração de Features
A extração via `cv2.moments` e `cv2.fitEllipse` produziu descritores robustos para os 12 blobs.
A razão eixo-maior/eixo-menor é um indicador de elongação independente de escala:
valores próximos de 1 indicam comprimidos redondos; valores altos revelam cápsulas ou comprimidos ovais.

### Distribuição das Features
A circularidade variou de **0,30** (blob mais irregular – possível cápsula alongada vista de lado)
a **0,83** (blob mais circular – comprimido redondo).
Nenhum blob atingiu o limiar clássico de 0,85, o que motivou a adoção de thresholds adaptados
à distribuição real dos dados (0,70 e 0,40 em vez de 0,85 e 0,70).
A solidez elevada (> 0,85 para a maioria) confirma que os objetos são sólidos, sem cavidades internas.
Os dois blobs com baixa solidez (< 0,88) e baixa circularidade (< 0,35) são provavelmente cápsulas
com formato muito alongado, onde a poligonal do contorno apresenta concavidades aparentes.

### Classificação por Regras
A combinação tamanho × forma produziu 5 classes distintas observadas na imagem:
- **Grande-Circular** (6 blobs): comprimidos redondos de maior área
- **Pequeno-Oval** (2 blobs): cápsulas pequenas
- **Pequeno-Irregular** (2 blobs): cápsulas muito alongadas
- **Médio-Circular** (1 blob): comprimido redondo médio
- **Médio-Oval** (1 blob): cápsula de tamanho médio

### Validação Manual
Os erros de área ficaram abaixo de **2,5 %** e os de perímetro abaixo de **3 %** para quatro dos
cinco blobs, valores típicos da medição manual por contagem de pixels no GIMP.
O blob #3 apresentou erro de perímetro ligeiramente maior (~2,5 %), atribuível à dificuldade de
traçar manualmente um contorno irregular com a régua.
Erros de área tendem a ser menores que os de perímetro porque contar pixels é mais direto do que
medir o comprimento de uma curva discreta.

### Limitações e Melhorias
- **Thresholds empíricos:** os limiares de classificação foram escolhidos pela inspeção visual
  da distribuição; k-means ou GMM produziriam fronteiras mais objetivas e reprodutíveis.
- **Sombras na imagem:** as sombras geradas pela IA expandem os contornos, elevando área
  e perímetro detectados acima do valor real do objeto físico.
- **Ampliação da validação:** incluir todos os 12 blobs na validação manual aumentaria
  a confiança estatística dos erros percentuais calculados.